In [1]:
#!/bin/bash
import specutils

In [2]:
import sys
import os
sys.path.insert(0,'..')
import simulacra.star
import simulacra.tellurics
from simulacra.star import PhoenixModel

import random
import numpy as np
import jax.numpy as jnp
import jax

import astropy.io.fits
import astropy.time as at

import astropy.units as u
import astropy.coordinates as coord
import astropy.constants as const

/ext3/miniforge3/lib/python3.9/site-packages/pysynphot/__init__.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/ext3/miniforge3/lib/python3.9/site-packages/pysynphot/locations.py:46: UserWarning: PYSYN_CDBS is undefined; functionality will be SEVERELY crippled.
  warnings.warn("PYSYN_CDBS is undefined; functionality will be SEVERELY "
/ext3/miniforge3/lib/python3.9/site-packages/pysynphot/locations.py:345: UserWarning: Extinction files not found in extinction
  warnings.warn('Extinction files not found in %s' % (extdir, ))


<h1>Simulacra: An Introduction to Simulating Spectrograph Data</h1>
This package should be used to simulate spectrographs by creating a star with a given flux, various transmission models (gas cell and tellurics), and a detector. Then simulating the detector at given start times for an exposure time.

### Theory
$$ F^{tot}_{ij} = Spline(x^{new}_j + \Delta(v_i) | x^s_{ij}, f^s_{ij}) \prod_{k \in T} Spline(x^{new}_j | x^k_{ij}, f^k_{ij})$$

$$ F^{lsf}_{ij} = k(R_j) \otimes F^{tot}_{ij}$$

$$ F^{det}_{\lambda ij} = Lanc(x^{det}_{j} | x^{new}_j, F^{lsf}_{ij}, a)$$ <br>

$$ N^{exp}_{ij} = F^{det}_{\lambda ij} t_{exp} A_{det}  \frac{R^2_{ste}}{D^2_{ste}} d \lambda_j$$

### Generating
Inside the detector, there is one stellar model that generates emission spectra, then a list of transmission models that are multiplied together to get this expected number of photons. The star model should output from `generate_spectra` three items: a 1d array of wavelength, a 1d array of flux in astropy units, and a 1d array of radial velocities at the given times. Once all the transmission grids are generated. Then lowest median grid spacing is used to a generate a new wavelength grid with equal spacing in log lambda space. Then all the transmission grids and spectra grid are interpolated onto the new grid using cubic splines. Finally these grids are multiplied element-wise to create the total theoretical flux. Now it's time to convolve the total flux with line spread function of the detector. This line spread function should be able to change over the pixel elements or the epoches. Once convolved, this is sampled at the `wave_grid` of the detector using Lanczos `a` Interpolation. This is the expected flux per pixel, which is convert to the expected number of photons using the above formula. d lambda is the difference over wavelength that is being measured at that ccd pixel. This can be set by the user but by default it takes element-wise differences and cates with the average. 

observing from my home.

In [3]:
latitude = 40.69140120265266
longitude = -73.91145844610405

dec_range = 23
ra, dec = np.random.uniform(0,360) * u.degree, np.random.uniform(max(latitude-dec_range,-90),min(latitude+dec_range,90)) * u.degree
obs = 'APO'
loc = coord.EarthLocation.from_geodetic(longitude,latitude,2)
target = coord.SkyCoord(ra,dec,frame='icrs')

Functions from the star module can be used to select times to view a given star from some observatory.

In [4]:
tstart = at.Time('2020-01-01T08:10:00.123456789',format='isot',scale='utc')
tend   = tstart + 360 * u.day
night_grid = simulacra.star.get_night_grid(loc,tstart,tend,steps_per_night=240)
possible_times, airmass = simulacra.star.get_realistic_times(target,loc,night_grid)

In [5]:
epoches = 60

Now we selected some random sample of these to observe at and the airmasses at those times

In [6]:
print(np.sort(airmass))
obs_ints = random.sample(range(len(airmass)),epoches)
sort_am = np.argsort(airmass)
obs_times, obs_airmass = possible_times[sort_am[:epoches]], airmass[sort_am[:epoches]]

[1.08387888 1.08388053 1.08388053 ... 2.99982106 2.99988782 2.99995165]


<h2>Tellurics Model</h2>
The tellurics model requires these airmasses at the time of observation. However each of the pressure, temperatures, and humidities can be set by the user after initialization. If a single value is passed that is used for every epoch. Or you can pass it an array of quantities of size equal to the number of epoches.

In [7]:
wave_min = 380*u.nm
wave_max = 690*u.nm
wave_padding = 20*u.nm
tellurics_model = simulacra.tellurics.TelFitModel(loc,wave_min,wave_max,wave_padding=wave_padding)

In [8]:
import matplotlib.pyplot as plt

Define some atmospheric parameters for the tellurics. These can either be constant over all time or an array with the same length as the number of epoches.

In [9]:
tellurics_model.pressure    = 870 *np.ones((epoches)) * u.hPa
tellurics_model.humidity    = 5.0 * np.ones((epoches))
tellurics_model.temperature = 315 * np.ones((epoches)) * u.Kelvin

<h2>Star Model</h2>
Here we define the star model with some temperature, distance, logg, and companion parameters. The logg, T, z, and alpha parameters must correspond to an appropriate atmosphere model from the PHOENIX libraray online. Then also give it some companion parameters that could affect its velocity. This is what we will be trying to find use jabble.

z is metallicity. And distance can be set to whatever you want.

In [10]:
logg = 1.0
T    = 4800
z    = -1.0
alpha= 0.4
distance  = 20 * u.pc
amplitude = 100 * u.m/u.s
period    = 7 * u.day
stellar_model = PhoenixModel(distance,alpha,z,T,logg,target,amplitude,period)

using saved wave file
your parameters: T 4800, logg 1.0, alpha 0.4, z -1.0
setting: T 4800, logg 1.0, alpha 0.4000000000000001, z -1.0
../data/stellar/PHOENIX/HiResFITS/PHOENIX-ACES-AGSS-COND-2011/Z-1.0.Alpha=+0.40/lte04800-1.00-1.0.Alpha=+0.40.PHOENIX-ACES-AGSS-COND-2011-HiRes.fits
using saved flux file
reading in ../data/stellar/PHOENIX/WAVE_PHOENIX-ACES-AGSS-COND-2011.fits


In [12]:
from simulacra.detector import Detector

<h2>Detector</h2>
Here we define our detector giving it an aperature area, resolution, dark current, read noise, and ccd efficiency. All of these can be except area can be given as an array of the same size as the wave_grid (eg. if the detector has varying resolution or noise levels)

In [13]:
JAX_PLATFORMS='cpu'

In [14]:
resolution = float(115_000)
area = np.pi*(2.5 * u.m/2)**2
# exp_times = [100,500] * u.second 
snrs = [350] * epoches
through_put  = 1e-8

dx = simulacra.star.delta_x(2*resolution)
size = int((np.log(wave_max.to(u.Angstrom).value) - np.log(wave_min.to(u.Angstrom).value))//dx)+1
x_grid = np.linspace(np.log(wave_min.to(u.Angstrom).value),np.log(wave_max.to(u.Angstrom).value),size,dtype=np.double)
wave_grid = np.exp(x_grid) * u.Angstrom

detector = Detector(stellar_model,resolution,loc,area,wave_grid,through_put,wave_padding)

In [15]:
detector.add_model(tellurics_model)

<h2>Gas Cell</h2>
Optionally, add the gas cell to the detector for simulations of the Keck HiRES spectrograph.

In [16]:
from simulacra.gascell import GasCellModel
gascell_model = GasCellModel('../data/gascell/keck_fts_inUse.idl')
# detector.add_model(gascell_model)

<h2>Simulator</h2>
Now comes the bulk of the work, run the simulation with the given transmission models, star, detector at the given times for some exposure times.

In [17]:
data = detector.simulate(obs_times,snrs=snrs)

surface flux: mean 1.33e+14 erg / (s cm3)	 median 1.16e+14 erg / (s cm3)
obs     flux: mean 9.57e-01 erg / (s cm3)	 median 8.37e-01 erg / (s cm3)
generating spectra...
humidity: 5.0
 pressure: 870.0
 temperature: 315.0
 lat: 40.69140120265267
 elevation: 0.00199999999832382
 freqmin(cm-1): 14084.507042253523
 freqmax(cm-1): 27777.777777777763
 angle: 22.688977535611166


Running exec: lblrtm




runlblrtm_v3.sh: 11: time: not found


TypeError: __init__() missing 1 required positional argument: 'cmd'

In [18]:
os.system('time')

sh: 1: time: not found


32512

In [ ]:
import h5py
filename = '/scratch/mdd423/simulacra/out/data_e{}_p{}_a{}_l{}-{}_salp{}_z{}_T{}_g{}_ra{}_dec{}_loc{}-{}.h5'.format(\
    epoches,period.to(u.day).value,amplitude.to(u.m/u.s).value,wave_min.to(u.nm).value,wave_max.to(u.nm).value,alpha,z,\
    T,logg,ra,dec,longitude,latitude)
data.to_h5(filename)

In [ ]:
import simulacra.dataset
data = simulacra.dataset.from_h5('/scratch/mdd423/simulacra/out/data_e60_p7.0_a100.0_l380.0-690.0_salp0.4_z-1.0_T4800_g1.0_ra183.33716301338228 deg_dec49.411485886697704 deg_loc-73.91145844610405-40.69140120265266.h5')
# airmass = tellurics_model.get_airmass(stellar_model,detector,obs_times)

In [ ]:
obs_times = data['data']['obs_times']
airmass = data['parameters']['tellurics']
rvs = data['data']['rvs']

In [ ]:
import matplotlib.pyplot as plt
import scipy.ndimage
def lin_normalize(y):
    y_min, y_max = y.min(), y.max()
    
    return (y - y_min)/(y_max - y_min)

def err_lin_normalize(yerr,y):
    y_min, y_max = y.min(), y.max()
    return yerr / (y_max - y_min)

In [ ]:
import matplotlib
matplotlib.rcParams['text.usetex'] = True

In [ ]:
plt_unit = u.Angstrom
sort_times = np.argsort(obs_times)

plt_width = 100
plt_mids = [3950,5170,6565]
plt_epoch = 40
fig, axes = plt.subplots(len(plt_mids),figsize=(8,3*len(plt_mids)),sharex=False,sharey=False,dpi=200,)


for i,mid in enumerate(plt_mids):

    axes[i].step(data['data']['wave'].to(u.Angstrom).value,lin_normalize(data['data']['flux'][plt_epoch,:]),'k',alpha=0.5,where='mid')
    axes[i].plot(data['theory']['total']['wave'][:].to(u.Angstrom).value,lin_normalize(data['theory']['star']['flux'][plt_epoch,:]),'r',alpha=0.3)
    axes[i].plot(data['theory']['total']['wave'][:].to(u.Angstrom).value,data['theory']['tellurics']['flux'][plt_epoch,:],'b',alpha=0.3)
    # axes[0].plot(data['theory']['total']['wave'][:].to(u.Angstrom).value,lin_normalize(data['theory']['lsf']['flux'][e_i,:]),'m',alpha=0.7)
    axes[i].set_xlim(mid - plt_width/2,mid + plt_width/2)

# axes[1].plot(data['theory']['total']['wave'][:].to(u.Angstrom).value,lin_normalize(data['theory']['star']['flux'][0,:]),'r',alpha=0.3)
# axes[1].plot(data['theory']['total']['wave'][:].to(u.Angstrom).value,data['theory']['tellurics']['flux'][0,:],'--b',alpha=0.3)
# axes[1].plot(data['theory']['total']['wave'][:].to(u.Angstrom).value,lin_normalize(data['theory']['lsf']['flux'][0,:]),'m',alpha=0.7)

# axes[0].set_xlim(6230,6250)
# axes[1].set_xlim(6280,6300)

fig.text(0.5, 0.06, 'Wavelength [$\AA$]', ha='center')
fig.text(0.04, 0.5, 'Normalized Flux', va='center', rotation='vertical')
plt.savefig('../out/dauntstar.png',bbox_inches='tight')
plt.show()

In [ ]:
# rvs = data['data']['rvs'].to(u.km/u.s).value
plt_epoches = np.array([np.argmin(airmass),np.argmax(airmass),np.argmin(rvs),np.argmax(rvs)])
plt_unit = u.Angstrom

sort_times = np.argsort(obs_times)
fig, axes = plt.subplots(len(plt_epoches),figsize=(6,3 * len(plt_epoches)),sharex=True,sharey=False,dpi=200)

for i,e_i in enumerate(plt_epoches):

    # print(data['theory']['tellurics']['wave'])
    # print('{:3.2e}'.format(np.mean(data['data']['flux'][i,:])),'{:3.2e}'.format(np.mean(data['data']['ferr'][i,:])))
    # axes[i].plot(data['data']['wave'][data['data']['mask'][i,:]].to(u.Angstrom).value,data['data']['flux'][i,data['data']['mask'][i,:]],'.r',alpha=0.5)
    axes[i].step(data['data']['wave'].to(u.Angstrom).value,lin_normalize(data['data']['flux'][e_i,:]),'k',alpha=0.5,where='mid')
    # axes[i].set_xlim(data['data']['wave'].to(u.Angstrom).value.min(),data['data']['wave'].to(u.Angstrom).value.max())
    axes[i].plot(data['theory']['total']['wave'][:].to(u.Angstrom).value,lin_normalize(data['theory']['star']['flux'][e_i,:]),'r',alpha=0.3)
    # axes[i].plot(data['theory']['total']['wave'][:].to(u.Angstrom).value,lin_normalize(data['theory']['total']['flux'][i,:]),'g',alpha=0.3,label='High Res')
    axes[i].plot(data['theory']['total']['wave'][:].to(u.Angstrom).value,data['theory']['tellurics']['flux'][e_i,:],'b',alpha=0.3)
    # axes[i].plot(data['theory']['total']['wave'][:].to(u.Angstrom).value,lin_normalize(data['theory']['lsf']['flux'][e_i,:]),'g',alpha=0.3,label='Low Res')
    # axes[i].plot(data['data']['wave'][:].to(u.Angstrom).value,lin_normalize(data['data']['flux_expected'][i,:]),'b',alpha=0.3)



    # axes[i].vlines(detector.wave_grid,-1,1)
    # axes[i].plot(data['theory']['interpolated']['total']['wave'][:].to(u.Angstrom).value,lin_normalize(data['theory']['interpolated']['tellurics']['flux'][i]),'b')
    # axes[i].plot(data['theory']['interpolated']['total']['wave'][:].to(u.Angstrom).value,lin_normalize(data['theory']['interpolated']['gascell']['flux'][i]),'g')

fig.text(0.5, 0.08, 'Wavelength [$\AA$]', ha='center')
fig.text(0.04, 0.5, 'Normalized Flux', va='center', rotation='vertical')
plt.xlim(6560,6570)
# plt.legend()
plt.show()

In [ ]:
times = obs_times
rv = data['data']['rvs'].to(u.km/u.s)
bc  = [target.radial_velocity_correction(obstime=time,location=loc).to(u.km/u.s).value for time in times] * u.km/u.s
print(rv,bc)
eprv = rv - bc


plt.figure(figsize=(10,3))
plt.title('EPRV')
v_unit = u.m/u.s
plt.plot(([(time - min(times)).to(u.day).value % period.to(u.day).value for time in times]),eprv.to(v_unit).value,'.r')
plt.ylabel('vel [{}]'.format(v_unit))
plt.xlabel('time [d]')
plt.show()

plt.figure(figsize=(10,3))
plt.title('RV')
v_unit = u.km/u.s
plt.plot(([(time - min(times)).to(u.day).value for time in times]),rv.to(v_unit).value,'.k')
plt.ylabel('vel [{}]'.format(v_unit))
plt.xlabel('time [d]')
plt.show()